# Hand Gesture — Model Compression Pipeline (PyTorch)
Steps 2–5: Train → Quantize → Prune → Benchmark

**Before running:** collect `gesture_data.csv` locally with `1_collect_data.py`, then upload it in Cell 2.

In [ ]:
# Cell 1 — Upload gesture_data.csv
from google.colab import files
print('Select your gesture_data.csv ...')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 2 — Imports (all pre-installed on Colab)
import os, time, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
print('PyTorch:', torch.__version__)

In [ ]:
# Cell 3 — Model definition
class GestureNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# Cell 4 — Load and split data
df = pd.read_csv('gesture_data.csv')
print(f'Samples: {len(df)}\n{df["label"].value_counts()}\n')

X  = df.drop('label', axis=1).values.astype('float32')
le = LabelEncoder()
y  = le.fit_transform(df['label'].values).astype('int64')
np.save('label_encoder_classes.npy', le.classes_)
print('Classes:', le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

In [ ]:
# Cell 5 — Train baseline model
EPOCHS, BATCH_SIZE, LR = 30, 32, 1e-3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

train_dl = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                      batch_size=BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                      batch_size=BATCH_SIZE)

model     = GestureNet(X_train.shape[1], len(le.classes_)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    correct = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train: optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            if train: loss.backward(); optimizer.step()
            correct += (out.argmax(1) == yb).sum().item()
    return correct / len(loader.dataset)

for epoch in range(1, EPOCHS+1):
    tr_acc  = run_epoch(model, train_dl, train=True)
    val_acc = run_epoch(model, test_dl,  train=False)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}  train={tr_acc*100:.1f}%  val={val_acc*100:.1f}%')

baseline_acc = run_epoch(model, test_dl, train=False)
print(f'\nBaseline accuracy: {baseline_acc*100:.2f}%')
torch.save({'model_state': model.state_dict(),
            'input_dim': X_train.shape[1],
            'num_classes': len(le.classes_)}, 'gesture_model.pt')
print('Saved gesture_model.pt')

In [ ]:
# Cell 6 — Dynamic int8 quantization
q_model = GestureNet(X_train.shape[1], len(le.classes_))
q_model.load_state_dict(model.cpu().state_dict())
q_model.eval()

q_model = torch.quantization.quantize_dynamic(q_model, {nn.Linear}, dtype=torch.qint8)
torch.save({'model_state': q_model.state_dict(),
            'input_dim': X_train.shape[1],
            'num_classes': len(le.classes_),
            'quantized': True}, 'gesture_model_quant.pt')
print(f'Quantized model saved: {os.path.getsize("gesture_model_quant.pt")/1024:.1f} KB')

In [ ]:
# Cell 7 — Pruning (50% sparsity) + fine-tune
SPARSITY, FT_EPOCHS = 0.5, 10
model.to(device)

# Re-load clean weights, then apply pruning
p_model = GestureNet(X_train.shape[1], len(le.classes_)).to(device)
ckpt = torch.load('gesture_model.pt', map_location=device)
p_model.load_state_dict(ckpt['model_state'])

for module in p_model.modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=SPARSITY)

p_optimizer = torch.optim.Adam(p_model.parameters(), lr=LR)
print(f'Fine-tuning pruned model for {FT_EPOCHS} epochs...')
for epoch in range(1, FT_EPOCHS+1):
    p_model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        p_optimizer.zero_grad()
        criterion(p_model(xb), yb).backward()
        p_optimizer.step()
    val = run_epoch(p_model, test_dl, train=False)
    print(f'Epoch {epoch:2d}  val={val*100:.1f}%')

# Make pruning permanent
for module in p_model.modules():
    if isinstance(module, nn.Linear):
        try: prune.remove(module, 'weight')
        except: pass

pruned_acc = run_epoch(p_model, test_dl, train=False)
print(f'\nPruned accuracy: {pruned_acc*100:.2f}%')
torch.save({'model_state': p_model.cpu().state_dict(),
            'input_dim': X_train.shape[1],
            'num_classes': len(le.classes_)}, 'gesture_model_pruned.pt')
print(f'Saved gesture_model_pruned.pt  ({os.path.getsize("gesture_model_pruned.pt")/1024:.1f} KB)')

In [ ]:
# Cell 8 — Benchmark all three models
MODELS = {
    'Baseline (float32)':  'gesture_model.pt',
    'Quantized (int8)':    'gesture_model_quant.pt',
    'Pruned (50% sparse)': 'gesture_model_pruned.pt',
}
LATENCY_RUNS = 100

def load_for_bench(path):
    ckpt = torch.load(path, map_location='cpu')
    m = GestureNet(ckpt['input_dim'], ckpt['num_classes'])
    if ckpt.get('quantized'):
        m = torch.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    return m

X_t = torch.from_numpy(X_test)
y_t = torch.from_numpy(y_test.astype('int64'))

results = []
for name, path in MODELS.items():
    m = load_for_bench(path)
    with torch.no_grad():
        acc = (m(X_t).argmax(1) == y_t).float().mean().item() * 100
    times = []
    with torch.no_grad():
        for i in range(LATENCY_RUNS):
            s = X_t[i % len(X_t)].unsqueeze(0)
            t0 = time.perf_counter()
            m(s)
            times.append((time.perf_counter()-t0)*1000)
    size_kb = os.path.getsize(path)/1024
    results.append({'Technique': name, 'Size (KB)': f'{size_kb:.1f}',
                    'Accuracy (%)': f'{acc:.2f}', 'Latency (ms)': f'{np.mean(times):.3f}'})
    print(f'{name}: {size_kb:.1f} KB | {acc:.2f}% | {np.mean(times):.3f} ms')

with open('compression_comparison_results.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['Technique','Size (KB)','Accuracy (%)','Latency (ms)'])
    w.writeheader(); w.writerows(results)

print('\n' + '='*62)
print(f'{"Technique":<25} {"Size (KB)":>10} {"Accuracy (%)":>13} {"Latency (ms)":>13}')
print('-'*62)
for r in results:
    print(f'{r["Technique"]:<25} {r["Size (KB)"]:>10} {r["Accuracy (%)":>13} {r["Latency (ms)"]:>13}')
print('='*62)

In [ ]:
# Cell 9 — Download all output files
from google.colab import files
for fname in ['compression_comparison_results.csv',
              'gesture_model.pt', 'gesture_model_quant.pt',
              'gesture_model_pruned.pt', 'label_encoder_classes.npy']:
    if os.path.exists(fname):
        files.download(fname)
        print('Downloaded:', fname)